In [1]:
import pandas as pd
import seaborn as sns
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
import os

from utils.utility import (
    load_parquet, load_csv, save_parquet,
    clean_iso_index, clean_iso_cols, clean_iso_col,
    pivot_binary, mineral_columns,
    build_and_save_graph, DATA_DIR, OUTPUT_DIR, PROCESSED_DIR,
)


In [2]:
# Load data
scores = load_parquet("notepad_03_criticality_scores.parquet").set_index("Mineral Code")
producer_matrix = load_parquet("mineral_country_matrix.parquet")
manufacturer_matrix = load_parquet("sector_country_matrix.parquet")

links = load_parquet("country_links.parquet")
dependency_matrix = load_parquet("dependency_matrix.parquet")

mineral_sector = load_parquet("mineral_sector_mat.parquet")
sector_country = load_parquet("sector_country_matrix.parquet")


In [3]:
# ISO cleaning

links["Country Code"] = links["Country Code"].astype(str).str.strip().str.upper()

dependency_matrix = clean_iso_index(dependency_matrix)
dependency_matrix = clean_iso_cols(dependency_matrix)

mineral_sector = clean_iso_index(mineral_sector)
mineral_sector = clean_iso_cols(mineral_sector)

sector_country = clean_iso_index(sector_country)
sector_country = clean_iso_cols(sector_country)


In [4]:
# Criticality Visualisations

# Load scoring table
# scores = scores.set_index("Mineral Code")

# Style
sns.set_theme(style="whitegrid", font_scale=1.2)

# 5. Save ranked table for dashboard use
ranked = scores.sort_values("Rank")
ranked.to_csv(PROCESSED_DIR / "notepad_04_ranked_table.csv")

# ranked.head(10)


# 1. Criticality ranking bar chart
plt.figure(figsize=(12, 10))
ranked = scores.sort_values("CriticalityScore", ascending=False)

sns.barplot(
    data=ranked,
    x="CriticalityScore",
    y=ranked.index,
    palette="viridis"
)

plt.title("Criticality Score Ranking")
plt.xlabel("Criticality Score")
plt.ylabel("Mineral Code")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "criticality_ranking.png", dpi=300, bbox_inches="tight")
plt.show()

# 2. Supply Risk vs Demand Importance scatter plot
plt.figure(figsize=(10, 8))

sns.scatterplot(
    data=scores,
    x="SupplyRisk",
    y="DemandImportance",
    hue="CriticalityScore",
    size="CriticalityScore",
    palette="viridis",
    sizes=(50, 300),
    legend="brief"
)

for mineral, row in scores.iterrows():
    plt.text(row["SupplyRisk"] + 0.005, row["DemandImportance"] + 0.005, mineral)

plt.title("Supply Risk vs Demand Importance")
plt.xlabel("Supply Risk")
plt.ylabel("Demand Importance")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "supply_vs_demand.png", dpi=300, bbox_inches="tight")

plt.show()


# 3. Quadrant chart (risk vs importance)
plt.figure(figsize=(10, 8))

median_supply = scores["SupplyRisk"].median()
median_demand = scores["DemandImportance"].median()

sns.scatterplot(
    data=scores,
    x="SupplyRisk",
    y="DemandImportance",
    hue="CriticalityScore",
    palette="viridis",
    s=120
)

# Quadrant lines
plt.axvline(median_supply, color="black", linestyle="--", alpha=0.7)
plt.axhline(median_demand, color="black", linestyle="--", alpha=0.7)

for mineral, row in scores.iterrows():
    plt.text(row["SupplyRisk"] + 0.005, row["DemandImportance"] + 0.005, mineral)

plt.title("Criticality Quadrants")
plt.xlabel("Supply Risk")
plt.ylabel("Demand Importance")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "criticality_quadrants.png", dpi=300, bbox_inches="tight")
plt.show()

# 4. Heatmap of all normalised metrics
norm_cols = [
    "HHI_norm",
    "ProducerDiversity_norm",
    "SectorUsage_norm",
    "EconomicImportance_norm",
    "SupplyRisk",
    "DemandImportance",
    "CriticalityScore"
]

plt.figure(figsize=(14, 10))
sns.heatmap(
    scores[norm_cols],
    cmap="viridis",
    linewidths=0.5
)

plt.title("Normalised Metrics Heatmap")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "metrics_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


C:\Users\rej4m\AppData\Local\Temp\ipykernel_8728\953808763.py:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


In [5]:
# world map builder
world = gpd.read_file(
    DATA_DIR / "ne_110m_admin_0_countries" / "ne_110m_admin_0_countries.shp"
).rename(columns={"ISO_A3": "Country Code"})

producer_counts = producer_matrix.sum(axis=1).rename("Mineral Production Count")
producer_geo = world.merge(producer_counts, on="Country Code", how="left")

plt.figure(figsize=(14, 8))
producer_geo.plot(
    column="Mineral Production Count",
    cmap="viridis",
    legend=True,
    missing_kwds={"color": "lightgrey"},
    edgecolor="black",
    linewidth=0.3
)
plt.title("Global Mineral Producer Countries")
plt.axis("off")
plt.savefig(OUTPUT_DIR / "producer_countries_map.png", dpi=300, bbox_inches="tight")
plt.show()

manufacturer_counts = manufacturer_matrix.sum(axis=0).rename("Sector Presence Count")
manufacturer_geo = world.merge(manufacturer_counts, on="Country Code", how="left")

plt.figure(figsize=(14, 8))
manufacturer_geo.plot(
    column="Sector Presence Count",
    cmap="plasma",
    legend=True,
    missing_kwds={"color": "lightgrey"},
    edgecolor="black",
    linewidth=0.3
)
plt.title("Global Manufacturer Countries")
plt.axis("off")
plt.savefig(OUTPUT_DIR / "manufacturer_countries_map.png", dpi=300, bbox_inches="tight")
plt.show()


<Figure size 1400x800 with 0 Axes>

<Figure size 1400x800 with 0 Axes>

In [6]:
# # Graph Builder
# def build_and_save_graph(G, name):
#     plt.figure(figsize=(14, 10))
#     pos = nx.spring_layout(G, k=0.4, iterations=50)
#
#     class_colors = {
#         "Integrated": "gold",
#         "Producer Only": "steelblue",
#         "Manufacturer Only": "firebrick",
#         "Neither": "grey"
#     }
#
#     node_colors = [class_colors[G.nodes[n]["dependency_class"]] for n in G.nodes()]
#     node_sizes = [800 * G.nodes[n]["capability"] for n in G.nodes()]
#     edge_weights = [G[u][v]["weight"] * 5 for u, v in G.edges()]
#
#     nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9)
#     nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.4, arrows=True, arrowstyle="-|>", arrowsize=12)
#     nx.draw_networkx_labels(G, pos, font_size=8)
#
#     plt.title(name)
#     plt.axis("off")
#     plt.tight_layout()
#     plt.savefig(OUTPUT_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
#     plt.show()
#
#     nx.write_gexf(G, PROCESSED_DIR / f"{name}.gexf")
#     print(f"Saved: {name}")


In [7]:
# Global Dependency Graph
G_global = nx.DiGraph()

for _, row in links.iterrows():
    G_global.add_node(
        row["Country Code"],
        country=row["Country"],
        producer_strength=row["Producer Strength"],
        manufacturer_strength=row["Manufacturer Strength"],
        capability=row["Capability Score"],
        dependency_class=row["Dependency Class"]
    )

for p in dependency_matrix.index:
    for m in dependency_matrix.columns:
        w = dependency_matrix.loc[p, m]
        if w > 0:
            G_global.add_edge(p, m, weight=w)

build_and_save_graph(G_global, "global_dependency_graph")
